# 07_03 Comparing fairly: one vocabulary for both poems

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → [model learns] → model writes → MCP tool
> ```
>
> You are here: **model learns**
>
> Project files this connects to:
>
> - `src/homer_ai/tokenizer.py`
> - `src/homer_ai/train_homer.py`
>
> **This notebook:** Two models trained on different poems are only comparable if they read text the same way. A shared vocabulary and a 2×2 test make the comparison fair.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why a tokenizer built from one poem can't read the other
- How a shared vocabulary fixes it
- How to run a fair 2×2 experiment: two models, each tested on both poems
- How to read the table without claiming too much

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **held-out text** | text a model never trained on, used only to test it (the validation set of 05_05) | the last 10% of each slice |
| **heatmap** | a grid of numbers drawn as colours, so you see the pattern at once | green = low loss, pink-red = high loss |

## 3. The idea

### ❓ Why can't I just train an Odyssey model and an Iliad model and compare their losses?

Because the text, the tokenizer and the test would all differ at once. Then a difference in the loss could come from any of them (remember the rule of [05_04](../05_training_on_the_odyssey/05_04_running_fair_experiments.ipynb): change **one** thing). We want the text to be the only difference. So the tokenizer must be the same, and the test text must be the same for both models.

Let's see what goes wrong when each poem has its own tokenizer.

In [ ]:
import torch

from homer_ai.lessons import HOMER, evaluate, fit, load_book_files, split_90_10
from homer_ai.tokenizer import CharTokenizer

SLICE = 10000  # characters of Book 1 we use: small, so every cell runs in seconds

odyssey = load_book_files(HOMER / "odyssey" / "greek")  # 24 books, one text each
iliad = load_book_files(HOMER / "iliad" / "greek")

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: what happens if I read the Iliad with the Odyssey's tokenizer?

We build a tokenizer from the Odyssey alone and use it on an Iliad verse. We saw in [07_01](07_01_the_corpus.ipynb) that the Iliad has six characters the Odyssey lacks. Where one appears, `<|unk|>` ([02_02](../02_text_to_numbers/02_02_special_tokens.ipynb)) takes its place.

👉 **Predict first:** the real verse below has `Ὕ` (capital upsilon with a mark). What will the Odyssey's tokenizer give back?

In [ ]:
odyssey_tok = CharTokenizer("\n".join(odyssey))
iliad_tok = CharTokenizer("\n".join(iliad))

verse = "οἵ τʼ Ἐλεῶνʼ εἶχον ἠδʼ Ὕλην καὶ Πετεῶνα,"  # Iliad, book 2
print("original:        ", verse)
print("Odyssey's tokens:", odyssey_tok.decode(odyssey_tok.encode(verse)))
print("Iliad's tokens:  ", iliad_tok.decode(iliad_tok.encode(verse)))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The Odyssey's tokenizer turns `Ὕ` into `<|unk|>`, while the Iliad's tokenizer reads the verse perfectly. One unknown letter is like a hole in a word. So how often does it happen over the whole Iliad?

</details>

In [ ]:
iliad_text = "".join(iliad)
unknown = [c for c in iliad_text if c not in odyssey_tok.char_to_id]
lines_hit = sum(
    any(c not in odyssey_tok.char_to_id for c in line)
    for line in iliad_text.splitlines()
)
print(f"unknown characters in the whole Iliad: {len(unknown)} of {len(iliad_text):,}")
all_verses = iliad_text.count("\n")
print(f"verses with at least one:             {lines_hit} of {all_verses:,}")
print("which ones:", " ".join(sorted(set(unknown))))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

That's 42 unknown characters out of 690,573, in 42 verses. Very rare, about 6 in 100,000. Small, but it isn't zero. The Odyssey's model can never read, or write, `Ξ` or `Ὕ`. The next problem is worse, because it's silent.

</details>

### ❓ Step 2: does the same letter get the same number in both tokenizers?

A token's number is its position in the sorted list of characters. If one tokenizer has an extra character in the list, every letter after it moves one place. A model trained with one tokenizer would then read the other's numbers as different letters.

👉 **Predict first:** the vocabularies are 169 and 171. Will most letters keep their number?

In [ ]:
moved = [
    c
    for c in odyssey_tok.chars
    if c in iliad_tok.char_to_id
    and odyssey_tok.char_to_id[c] != iliad_tok.char_to_id[c]
]
sizes = (odyssey_tok.vocab_size, iliad_tok.vocab_size)
print("vocabulary sizes:", sizes[0], "(Odyssey),", sizes[1], "(Iliad)")
print("characters in both, with a different number:", len(moved))
for c in moved[:3]:
    ids = (odyssey_tok.char_to_id[c], iliad_tok.char_to_id[c])
    print(f"  {c}  Odyssey {ids[0]}  Iliad {ids[1]}")
print("<|unk|>:       ", odyssey_tok.unk_id, "(Odyssey)", iliad_tok.unk_id, "(Iliad)")
print("<|endoftext|>: ", odyssey_tok.eot_id, "(Odyssey)", iliad_tok.eot_id, "(Iliad)")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

54 characters that both poems use have a **different number** in the two tokenizers. `Ο` is 19 in the Odyssey's and 20 in the Iliad's, because the Iliad has an extra character (`Ξ`) earlier in the sorted list, and every letter after it moves one place. The special tokens move too (`<|unk|>` is 167 against 169). So a model trained with one tokenizer would read the other's numbers as the wrong letters. And the two vocabularies have different sizes (169 and 171), which means the two models wouldn't even have the same shape. A comparison built on them would be unfair twice over.

</details>

### ❓ What's the fix?

One **shared vocabulary**. We build a single tokenizer from both poems, and give it to both models. Every letter gets one number, for every model. The special tokens still sit at the end (`<|unk|>`, `<|endoftext|>`), and this time in the same place for everyone.

```
Odyssey text ┐
              ├──▶ ONE tokenizer ──▶ Odyssey model
Iliad text   ┘                  └──▶ Iliad model
```

👉 **Predict first:** how many unknown characters will the shared tokenizer find in each poem?

In [ ]:
import math

shared = CharTokenizer("\n".join(odyssey + iliad))
print("shared vocabulary size:", shared.vocab_size)

for name, books in (("Odyssey", odyssey), ("Iliad", iliad)):
    text = "".join(books)
    unknown = sum(c not in shared.char_to_id for c in text)
    print(f"{name}: {unknown} unknown characters of {len(text):,}")
guess = math.log(shared.vocab_size)
print(f"random guessing now costs ln(175) = {guess:.3f} for every model")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A vocabulary of 175 (173 characters and the two special ones), and **0** unknown characters in either poem. There's one cost. The Odyssey model gets table rows for six Iliad-only characters it never sees. Those rows stay at their random start, just like `<|unk|>` in [02_02](../02_text_to_numbers/02_02_special_tokens.ipynb), and the model learns never to guess them.

</details>

### ❓ Step 3: how do I set up the fair experiment?

We write the rules down first, like the results log of [05_04](../05_training_on_the_odyssey/05_04_running_fair_experiments.ipynb):

| | Same for every run | The ONE thing that differs |
|---|---|---|
| tokenizer | the shared one (175) | |
| text size | the first 10,000 characters of Book 1 of each poem | |
| split | first 90% train, last 10% held out (05_05) | |
| model, epochs, lr | `HomerGreekLLM`, 4 epochs, 0.01 | |
| seed | 0 and 1 (two runs, then the average) | |
| **training poem** | | **Odyssey for model A, Iliad for model B** |

Each model is then tested on both held-out parts (1,000 characters each). Short slices and 4 epochs keep it to a few seconds per model.

👉 **Predict first:** on the Odyssey's held-out part, which model will have the lower loss? And on the Iliad's?

In [ ]:
def book_1_split(books):
    data = torch.tensor(shared.encode(books[0][:SLICE]))
    return split_90_10(data)  # (train, held-out)


odyssey_train, odyssey_val = book_1_split(odyssey)
iliad_train, iliad_val = book_1_split(iliad)
print("Odyssey: train", len(odyssey_train), "held-out", len(odyssey_val))
print("Iliad:   train", len(iliad_train), "held-out", len(iliad_val))

SEEDS = (0, 1)
tests = {"Odyssey held-out": odyssey_val, "Iliad held-out": iliad_val}
trained_on = {"Odyssey": odyssey_train, "Iliad": iliad_train}

table = {}  # (trained on, tested on) -> average loss over the seeds
for poem, train_data in trained_on.items():
    runs = [fit(train_data, shared.vocab_size, seed) for seed in SEEDS]
    for test, val_data in tests.items():
        table[(poem, test)] = sum(evaluate(m, val_data) for m in runs) / len(runs)

print()
print(f"{'':18} {'tested on Odyssey':>18} {'tested on Iliad':>16}")
for poem in trained_on:
    print(
        f"trained on {poem:7} {table[(poem, 'Odyssey held-out')]:>18.3f} "
        f"{table[(poem, 'Iliad held-out')]:>16.3f}"
    )

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The Odyssey model scores 2.811 on the Odyssey's held-out part and 3.026 on the Iliad's. The Iliad model scores 2.880 and 2.873. Each number is the average of two runs (seeds 0 and 1). Random guessing would give 5.165, so both models learned something about Greek. The picture shows the pattern.

</details>

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import GOOD_BAD, INK, studio_axes, studio_figure, text_on

values = [[table[(p, t)] for t in tests] for p in trained_on]
low = min(min(row) for row in values) - 0.05
high = max(max(row) for row in values) + 0.05

fig = studio_figure("fair comparison", "Loss: own poem against the other poem", (8, 6))
ax = studio_axes(fig, (0.2, 0.12, 0.7, 0.62))
ax.imshow(values, cmap=GOOD_BAD.reversed(), vmin=low, vmax=high, aspect="auto")
for r, row in enumerate(values):
    for c, value in enumerate(row):
        middle = abs(2 * (value - low) / (high - low) - 1)  # far from the middle
        ax.text(c, r, f"{value:.2f}", ha="center", va="center", fontsize=20,
                fontweight="bold", family="monospace", color=text_on(middle, 1))
ax.set_xticks([0, 1], ["Odyssey\nheld-out", "Iliad\nheld-out"])
ax.set_yticks([0, 1], ["trained on\nOdyssey", "trained on\nIliad"])
ax.tick_params(axis="both", labelsize=12, labelcolor=INK, length=0)
ax.grid(visible=False)
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Green is the lower (better) loss and pink-red the higher. In the left column the Odyssey model is greener than the Iliad model (2.81 against 2.88). In the right column the Iliad model is much greener (2.87 against 3.03).

</details>

### ❓ What does the table really say?

Read **down each column** (the same test text):

- On the Odyssey's held-out part, the Odyssey model is better: 2.811 against 2.880 (a gap of 0.07).
- On the Iliad's held-out part, the Iliad model is better: 2.873 against 3.026 (a gap of 0.15).

So each model does a little better on its own poem, as we expected. But a few warnings before you trust that:

- The gaps are small, the model is tiny, and we have two seeds and only 1,000 held-out characters per poem. So treat the 0.07 as fragile. Set `SLICE = 8000` in the first cell and see whether the order in the Odyssey column holds (that's a fair experiment too, because you change only one thing).
- Don't read across a row. The Iliad's held-out part is a different text from the Odyssey's, and it may simply be harder or easier. Only differences down a column are fair.
- Our model has a window of 24 characters and one attention head, so it can only learn short letter patterns. Both poems share such patterns, which is why the models aren't far apart.

## 5. Recap

- Separate tokenizers have different sizes and move letters to other numbers, so models built on them can't be compared. One shared vocabulary fixes it.
- A fair 2×2 table keeps the same slices, seed, epochs and tokenizer. The training poem is the only change, and each model is tested on both held-out parts.
- Report small, noisy gaps as small and noisy.

**Next:** [07_04: Combining both](07_04_combining_both.ipynb).